In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange, repeat

class PatchEmbedding3D(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size):
        super().__init__()
        self.proj = nn.Conv3d(in_channels, embed_dim, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        # x: (B, C, D1, D2, D3)
        x = self.proj(x)  # (B, embed_dim, D1//patch_size, D2//patch_size, D3//patch_size)
        x = rearrange(x, 'b c d1 d2 d3 -> b (d1 d2 d3) c')  # (B, N_patches, embed_dim)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B, num_heads, N, head_dim)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # (B, N, C)
        x = self.proj(x)
        x = self.dropout(x)
        return x

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim, int(dim * mlp_ratio), dropout)
        
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Vision3DTransformer(nn.Module):
    def __init__(
        self,
        in_channels,
        patch_size,
        embed_dim,
        depth,
        num_heads,
        mlp_ratio=4,
        dropout=0.0,
        cross_attn_dim=None,  # Dimension for cross-attention with LLM
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding3D(in_channels, embed_dim, patch_size)
        
        # Position embedding
        # self.pos_embed = nn.Parameter(torch.zeros(1, 1000, embed_dim))  # Max 1000 patches as placeholder
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        
        # Project to cross-attention dimension if needed
        self.cross_attn_dim = cross_attn_dim
        if cross_attn_dim is not None and cross_attn_dim != embed_dim:
            self.cross_attn_proj = nn.Linear(embed_dim, cross_attn_dim)
        else:
            self.cross_attn_proj = nn.Identity()
            
        # Initialize weights
        # self._init_weights()
    
    # def _init_weights(self):
        # Initialize position embeddings
        # nn.init.normal_(self.pos_embed, std=0.02)
    
    def forward(self, x):
        # x: (B, C, D1, D2, D3)
        B = x.shape[0]
        
        # Patch embeddings
        x = self.patch_embed(x)  # (B, N_patches, embed_dim)
        N_patches = x.shape[1]
        
        # Add position embeddings
        # pos_embed = self.pos_embed[:, :N_patches, :]
        # x = x + pos_embed
        
        # Apply transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Apply final normalization
        x = self.norm(x)
        
        # Project to cross-attention dimension if needed
        x = self.cross_attn_proj(x)
        
        return x  # (B, N_patches, cross_attn_dim or embed_dim)

# Example usage
def create_3d_vit(n_samp, n_channels, dim1, dim2, dim3, patch_size=4, embed_dim=768, depth=12, num_heads=12, cross_attn_dim=1024):
    model = Vision3DTransformer(
        in_channels=n_channels,
        patch_size=patch_size,
        embed_dim=embed_dim,
        depth=depth,
        num_heads=num_heads,
        dropout=0.1,
        cross_attn_dim=cross_attn_dim
    )
    
    # Example forward pass
    x = torch.randn(n_samp, n_channels, dim1, dim2, dim3)
    features = model(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output features shape: {features.shape}")
    
    return model, features

# Example: create_3d_vit(8, 3, 32, 32, 32)

In [9]:
create_3d_vit(14, 12, 8, 8, 8)



Input shape: torch.Size([14, 12, 8, 8, 8])
Output features shape: torch.Size([14, 8, 1024])


(Vision3DTransformer(
   (patch_embed): PatchEmbedding3D(
     (proj): Conv3d(12, 768, kernel_size=(4, 4, 4), stride=(4, 4, 4))
   )
   (blocks): ModuleList(
     (0-11): 12 x TransformerBlock(
       (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
       (attn): MultiHeadAttention(
         (qkv): Linear(in_features=768, out_features=2304, bias=True)
         (proj): Linear(in_features=768, out_features=768, bias=True)
         (dropout): Dropout(p=0.1, inplace=False)
       )
       (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
       (ffn): FeedForward(
         (net): Sequential(
           (0): Linear(in_features=768, out_features=3072, bias=True)
           (1): GELU(approximate='none')
           (2): Dropout(p=0.1, inplace=False)
           (3): Linear(in_features=3072, out_features=768, bias=True)
           (4): Dropout(p=0.1, inplace=False)
         )
       )
     )
   )
   (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
   (cro